# AgroConecta Mantaro: Entrenamiento de Regresión Lineal Múltiple
## Estimación de Tiempos de Viaje de Transporte Agrícola en el Valle del Mantaro (Junín)

**Contexto:**
- Cátedra de Machine Learning (UPLA - Ing. Jowel Cabrera).
- Dataset oficial basado en la red vial de Junín (OSRM) y la normativa del MTC (**D.S. N.° 020-2021-MTC** / Provías Nacional).
- **Objetivo:** Predecir con precisión el tiempo total de viaje ($y$) a partir de la distancia por carretera ($X_1$), el peso de carga consolidada ($X_2$) y el estado de congestión en hora punta hacia Huancayo ($X_3$).

$$\hat{y} = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \beta_3 X_3$$

Solución analítica de Mínimos Cuadrados Ordinarios (OLS):
$$\hat{\beta} = (X^T X)^{-1} X^T y$$

In [ ]:
# 1. Importación de Librerías Científicas
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

sns.set_theme(style="whitegrid")
print("Librerías importadas correctamente.")

### 2. Carga del Dataset Real del Valle del Mantaro
El dataset contiene **576 registros de viajes de carga** entre los distritos agrícolas (Jauja, Concepción, Orcotuna, Sicaya, Chupaca, Mito, Saño, etc.) hacia el **Gran Mercado Mayorista de Huancayo**.

In [ ]:
# Carga desde URL de GitHub o archivo local
url = "https://raw.githubusercontent.com/Learning202413/AgroConecta-Mantaro/master/data/dataset_transporte_mantaro.csv"
try:
    df = pd.read_csv(url)
    print("Cargado exitosamente desde GitHub.")
except:
    df = pd.read_csv("../data/dataset_transporte_mantaro.csv")
    print("Cargado desde ruta local.")

print(f"Total de observaciones: {len(df)} filas, {len(df.columns)} columnas.")
df.head()

### 3. Análisis Exploratorio de Datos (EDA)

In [ ]:
# Resumen estadístico de las variables principales
features = ['distancia_km', 'peso_carga_ton', 'hora_punta', 'tiempo_total_viaje_min']
df[features].describe()

In [ ]:
# Matriz de Correlación de Pearson
plt.figure(figsize=(8, 6))
corr = df[features].corr()
sns.heatmap(corr, annot=True, cmap="Greens", fmt=".3f", cbar=True)
plt.title("Matriz de Correlación - Variables de Transporte Mantaro")
plt.show()

### 4. Partición Train / Test (80% Entrenamiento, 20% Prueba)

In [ ]:
X = df[['distancia_km', 'peso_carga_ton', 'hora_punta']].values
y = df['tiempo_total_viaje_min'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print(f"Conjunto de entrenamiento: {X_train.shape[0]} muestras")
print(f"Conjunto de evaluación:   {X_test.shape[0]} muestras")

### 5. Entrenamiento por Ecuación Normal de Mínimos Cuadrados (OLS Exacto)
Implementamos la fórmula matricial explicada por el Ing. Jowel Cabrera:
$$\hat{\beta} = (X^T X)^{-1} X^T y$$

In [ ]:
# Añadimos la columna de unos para el término independiente (intercepto beta_0)
X_train_b = np.c_[np.ones((len(X_train), 1)), X_train]
X_test_b = np.c_[np.ones((len(X_test), 1)), X_test]

# Solución analítica OLS cerrada
beta = np.linalg.inv(X_train_b.T @ X_train_b) @ (X_train_b.T @ y_train)

print("=== COEFICIENTES OLS OBTENIDOS ===")
print(f"beta_0 (Intercepto):      {beta[0]:.4f} min")
print(f"beta_1 (Distancia km):    {beta[1]:.4f} min/km")
print(f"beta_2 (Carga toneladas): {beta[2]:.4f} min/ton")
print(f"beta_3 (Hora Punta):      {beta[3]:.4f} min")
print(f"\nFórmula: T_viaje = {beta[0]:.2f} + {beta[1]:.2f}*Dist + {beta[2]:.2f}*Carga + {beta[3]:.2f}*Punta")

### 6. Validación cruzada con Scikit-Learn
Verificamos que nuestra solución OLS pura coincide al 100% con la librería estándar de la industria.

In [ ]:
model_sk = LinearRegression()
model_sk.fit(X_train, y_train)

print("=== COMPARACIÓN OLS MANUAL VS SCIKIT-LEARN ===")
print(f"Intercepto: OLS = {beta[0]:.4f} | Scikit-Learn = {model_sk.intercept_:.4f}")
for i, col in enumerate(['Distancia', 'Carga (t)', 'Hora Punta']):
    print(f"{col:12}: OLS = {beta[i+1]:.4f} | Scikit-Learn = {model_sk.coef_[i]:.4f}")

### 7. Evaluación de Métricas de Precisión en Test Set

In [ ]:
y_pred = X_test_b @ beta

r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)

print("=== MÉTRICAS DE EVALUACIÓN EN EL CONJUNTO DE PRUEBA ===")
print(f"Coeficiente de Determinación (R²): {r2:.4f} ({r2*100:.2f}% de varianza explicada)")
print(f"Error Cuadrático Medio (MSE):       {mse:.2f} min²")
print(f"Raíz del MSE (RMSE):               {rmse:.2f} min")
print(f"Error Absoluto Medio (MAE):        {mae:.2f} min")

### 8. Gráficos de Diagnóstico: Real vs Predicho y Residuos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Gráfico de dispersión Real vs Predicho
axes[0].scatter(y_test, y_pred, color='#15803d', alpha=0.7, edgecolors='k')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Ajuste Perfecto 1:1')
axes[0].set_xlabel('Tiempo Real Observado (min)')
axes[0].set_ylabel('Tiempo Predicho por el Modelo (min)')
axes[0].set_title(f'Valores Reales vs Predichos (R² = {r2:.3f})')
axes[0].legend()

# 2. Histograma de Residuos (Errores)
residuals = y_test - y_pred
sns.histplot(residuals, kde=True, ax=axes[1], color='#1e40af')
axes[1].axvline(0, color='r', linestyle='--')
axes[1].set_xlabel('Residuo e = y_real - y_pred (min)')
axes[1].set_title(f'Distribución de Residuos (Media = {np.mean(residuals):.2f})')

plt.tight_layout()
plt.show()

### 9. Demostración de Inferencia en Tiempo Real para el Valle del Mantaro
Ejemplo: Un camión que recoge **3.5 toneladas** de papa en **Orcotuna (16.4 km)** y viaja al **Mercado Mayorista de Huancayo** durante **hora punta** con **1 parada de estiba (12 min)**:

In [ ]:
distancia_ejemplo = 16.4
peso_ton_ejemplo = 3.5
hora_punta_ejemplo = 1
estiba_min = 12.0

tiempo_viaje_estimado = beta[0] + beta[1] * distancia_ejemplo + beta[2] * peso_ton_ejemplo + beta[3] * hora_punta_ejemplo
tiempo_total_estimado = tiempo_viaje_estimado + estiba_min

print("=== ESTIMACIÓN DETALLADA ===")
print(f"Distancia: {distancia_ejemplo} km")
print(f"Carga:     {peso_ton_ejemplo} toneladas ({peso_ton_ejemplo*1000:.0f} kg)")
print(f"Condición: Hora punta en El Tambo / Huancayo")
print(f"Tiempo de viaje en carretera: {tiempo_viaje_estimado:.1f} minutos")
print(f"Tiempo de estiba / carga:     {estiba_min:.1f} minutos")
print(f"TIEMPO TOTAL ESTIMADO:        {tiempo_total_estimado:.0f} minutos ({tiempo_total_estimado/60:.2f} horas)")